# Agentes Especializados: Research, Financial y Code Review

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/10-agentes-especializados.ipynb)

Tres agentes especializados con herramientas y prompts optimizados para su dominio.

In [ ]:
!pip install anthropic duckduckgo-search requests -q

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

## 1. Research Agent: investigación web y síntesis

In [ ]:
from duckduckgo_search import DDGS

def buscar_web(query: str, max_resultados: int = 5) -> list[dict]:
    """Busca en DuckDuckGo y devuelve resultados."""
    resultados = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_resultados):
            resultados.append({'titulo': r['title'], 'url': r['href'], 'snippet': r['body']})
    return resultados

TOOLS_RESEARCH = [
    {
        'name': 'buscar_web',
        'description': 'Busca información en internet sobre un tema',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Consulta de búsqueda'},
                'max_resultados': {'type': 'integer', 'default': 5},
            },
            'required': ['query'],
        },
    },
]

def research_agent(pregunta: str) -> str:
    """Agente de investigación que busca y sintetiza información."""
    mensajes = [{'role': 'user', 'content': pregunta}]

    for _ in range(5):
        response = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=2048,
            system='Eres un investigador experto. Busca información y sintetiza una respuesta bien fundamentada.',
            tools=TOOLS_RESEARCH,
            messages=mensajes,
        )

        mensajes.append({'role': 'assistant', 'content': response.content})
        tool_uses = [b for b in response.content if b.type == 'tool_use']

        if not tool_uses:
            return next((b.text for b in response.content if b.type == 'text'), '')

        resultados = []
        for tu in tool_uses:
            if tu.name == 'buscar_web':
                resultados_busqueda = buscar_web(tu.input['query'], tu.input.get('max_resultados', 5))
                resultados.append({'type': 'tool_result', 'tool_use_id': tu.id, 'content': json.dumps(resultados_busqueda, ensure_ascii=False)})

        mensajes.append({'role': 'user', 'content': resultados})

    return 'Límite de iteraciones alcanzado'

# Ejecutar research agent
respuesta = research_agent('¿Cuáles son las últimas novedades del protocolo A2A de Google para agentes IA en 2025?')
print(respuesta[:500])

## 2. Financial Agent: análisis de mercado (solo lectura)

In [ ]:
# Nota: el Financial Agent SOLO analiza datos. Nunca ejecuta trades.

def obtener_datos_financieros_mock(ticker: str) -> dict:
    """Mock de datos financieros (en producción: yfinance)"""
    datos_mock = {
        'AAPL': {'precio': 185.50, 'cambio_dia': 1.2, 'pe_ratio': 28.5, 'market_cap': '2.9T', 'sector': 'Technology'},
        'MSFT': {'precio': 415.30, 'cambio_dia': 0.8, 'pe_ratio': 35.2, 'market_cap': '3.1T', 'sector': 'Technology'},
        'GOOGL': {'precio': 175.20, 'cambio_dia': -0.5, 'pe_ratio': 22.1, 'market_cap': '2.2T', 'sector': 'Technology'},
    }
    return datos_mock.get(ticker, {'error': f'Ticker {ticker} no encontrado'})

TOOLS_FINANCIAL = [
    {
        'name': 'obtener_datos_accion',
        'description': 'Obtiene precio y métricas de una acción. SOLO para análisis, nunca para ejecutar trades.',
        'input_schema': {
            'type': 'object',
            'properties': {'ticker': {'type': 'string', 'description': 'Símbolo bursátil (ej: AAPL)'}},
            'required': ['ticker'],
        },
    },
]

def financial_agent(consulta: str) -> str:
    mensajes = [{'role': 'user', 'content': consulta}]
    system = """Eres un analista financiero. Analizas datos del mercado y proporcionas insights.
IMPORTANTE: Solo analizas datos. NUNCA ejecutas trades, operaciones o movimientos de dinero.
Incluye siempre un disclaimer: 'Este análisis es solo informativo, no constituye asesoramiento financiero.'"""

    for _ in range(5):
        response = client.messages.create(
            model='claude-sonnet-4-6', max_tokens=1024,
            system=system, tools=TOOLS_FINANCIAL, messages=mensajes,
        )
        mensajes.append({'role': 'assistant', 'content': response.content})
        tool_uses = [b for b in response.content if b.type == 'tool_use']
        if not tool_uses:
            return next((b.text for b in response.content if b.type == 'text'), '')
        resultados = []
        for tu in tool_uses:
            datos = obtener_datos_financieros_mock(tu.input['ticker'])
            resultados.append({'type': 'tool_result', 'tool_use_id': tu.id, 'content': json.dumps(datos)})
        mensajes.append({'role': 'user', 'content': resultados})

    return 'Límite de iteraciones'

print(financial_agent('Compara Apple y Microsoft. ¿Cuál tiene mejor valoración actualmente?')[:400])

## 3. Code Review Agent

In [ ]:
def code_review_agent(codigo: str, lenguaje: str = 'python') -> dict:
    """Revisa código y devuelve un informe estructurado."""
    tool_review = {
        'name': 'generar_informe_revision',
        'description': 'Genera el informe de revisión de código',
        'input_schema': {
            'type': 'object',
            'properties': {
                'calidad_general': {'type': 'integer', 'minimum': 1, 'maximum': 10},
                'bugs': {'type': 'array', 'items': {'type': 'string'}},
                'mejoras': {'type': 'array', 'items': {'type': 'string'}},
                'seguridad': {'type': 'array', 'items': {'type': 'string'}},
                'resumen': {'type': 'string'},
            },
            'required': ['calidad_general', 'bugs', 'mejoras', 'seguridad', 'resumen'],
        },
    }

    response = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=1024,
        system=f'Eres un experto en {lenguaje}. Revisa el código con ojo crítico.',
        tools=[tool_review],
        tool_choice={'type': 'tool', 'name': 'generar_informe_revision'},
        messages=[{'role': 'user', 'content': f'Revisa este código {lenguaje}:\n```{lenguaje}\n{codigo}\n```'}],
    )

    tu = next(b for b in response.content if b.type == 'tool_use')
    return tu.input

codigo_a_revisar = '''
def dividir(a, b):
    return a / b

def procesar_lista(items):
    resultado = []
    for i in range(len(items)):
        resultado.append(items[i] * 2)
    return resultado
'''

informe = code_review_agent(codigo_a_revisar)
print(f'Calidad: {informe["calidad_general"]}/10')
print(f'Bugs: {informe["bugs"]}')
print(f'Mejoras: {informe["mejoras"]}')
print(f'Resumen: {informe["resumen"]}')